In [9]:
# !pip install findspark

## Пререквизиты

Код в этом ноутбуке будет работать с MLFlow Tracking Server, который запущен на удаленной виртуальной машине. Кроме того, артефакты моделирования сохранятся в S3-бакет в YC. Поэтому **прежде чем выполнять ячейки с кодом**, убедитесь, что установлены следующие переменные окружения:

```bash
MLFLOW_S3_ENDPOINT_URL=https://storage.yandexcloud.net/
MLFLOW_TRACKING_URI=http://<ip вашей виртуалки с MLFlow>:8000
AWS_ACCESS_KEY_ID=<id вашего ключа>
AWS_SECRET_ACCESS_KEY=<ваш секретный ключ>
```

Установить переменные я рекомендую так:

1. Создаете файл (в той же директории, откуда запускаете `jupyter`) `.env`.
2. Записываете в файл `env` следующее содержание:
   
```bash
MLFLOW_S3_ENDPOINT_URL=https://storage.yandexcloud.net/
MLFLOW_TRACKING_URI=http://<ip вашей виртуалки с MLFlow>:8000
AWS_ACCESS_KEY_ID=<id вашего ключа>
AWS_SECRET_ACCESS_KEY=<ваш секретный ключ>
```
3. Устанавливаете пакет `python-dotenv`

```bash
pip install python-dotenv`
```

4. Выполняете следующую ячейку с кодом:

In [1]:
%load_ext dotenv
%dotenv

In [2]:
import os
import findspark
import argparse
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, IntegerType
from pyspark.sql.functions import col, split, month, dayofmonth, dayofweek, hour, minute

from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.ml import Pipeline

import os
import logging
from functools import partial

import mlflow
import numpy as np
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType
from pyspark.sql.functions import month, dayofmonth, dayofweek, hour, minute

## Инициализируем spark

In [4]:
    # .config("spark.hadoop.fs.s3a.access.key", os.environ['AWS_ACCESS_KEY_ID']) \
    # .config("spark.hadoop.fs.s3a.secret.key", os.environ['AWS_SECRET_ACCESS_KEY']) \
    # .config('spark.hadoop.fs.s3.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem') \
    # .config('spark.yarn.dist.archives', 's3a://pyspark-venvs/mlflow-hyperopt-dataproc-2.1.18.tar.gz#venv')\

In [3]:
os.environ['PYSPARK_PYTHON'] = './venv/bin/python'
spark = SparkSession\
    .builder\
    .master('local[*]')\
    .appName('Spark ML Research')\
    .config('spark.sql.repl.eagerEval.enabled', True) \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/01/26 17:39:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/01/26 17:40:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/01/26 17:40:00 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/01/26 17:40:00 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [4]:
spark

## Считываем данные 

In [5]:
data_path = '/Users/admin/PycharmProjects/fraud_detection1/fraud_detection/data'
df = spark.read.text(f"{data_path}/*.txt")

def clean_data(df):
    df = df.na.drop(how="all")
    df = df.filter(~df.value.startswith("#"))

    columns = ["transaction_id", "tx_datetime", "customer_id", "terminal_id",
               "tx_amount", "tx_time_seconds", "tx_time_days", "tx_fraud", "tx_fraud_scenario"]
    
    columns_to_cast = ["tx_amount", "tx_time_seconds", "tx_time_days", "tx_fraud"]
    
    df = df.selectExpr("split(value, ',') as columns")
    df = df.selectExpr(*[f"columns[{i}] as {col}" for i, col in enumerate(columns)])

    for column in columns_to_cast:
        df = df.withColumn(column, F.col(column).cast(FloatType()))
    
    columns_to_rename = {cl: cl.replace('tx_', '') for cl in df.columns if cl.startswith('tx_')}
    for old_col, new_col in columns_to_rename.items():
        df = df.withColumnRenamed(old_col, new_col)

    df = df.withColumn('month', month(F.col('datetime')).cast('float')) \
        .withColumn('day', dayofmonth(F.col('datetime')).cast('float')) \
        .withColumn('day_of_week', dayofweek(F.col('datetime')).cast('float')) \
        .withColumn('hour', hour(F.col('datetime')).cast('float')) \
        .withColumn('minute', minute(F.col('datetime')).cast('float'))

    df = df.na.drop()
    df = df.withColumn('fraud', F.col('fraud').cast('float'))

    return df

df = clean_data(df)
df.limit(5)

transaction_id,datetime,customer_id,terminal_id,amount,time_seconds,time_days,fraud,fraud_scenario,month,day,day_of_week,hour,minute
0,2019-08-22 06:51:03,0,711,70.91,24663.0,0.0,0.0,0,8.0,22.0,5.0,6.0,51.0
1,2019-08-22 05:10:37,0,0,90.55,18637.0,0.0,0.0,0,8.0,22.0,5.0,5.0,10.0
2,2019-08-22 19:05:33,0,753,35.38,68733.0,0.0,0.0,0,8.0,22.0,5.0,19.0,5.0
3,2019-08-22 07:21:33,0,0,80.41,26493.0,0.0,0.0,0,8.0,22.0,5.0,7.0,21.0
4,2019-08-22 09:06:17,1,981,102.83,32777.0,0.0,0.0,0,8.0,22.0,5.0,9.0,6.0


## Конструируем пайплайн обработки данных

In [6]:
TRAIN_COLUMNS = [
                'amount',
                'month',
                'day',
                'day_of_week',
                'minute'
    ]
df.select(*TRAIN_COLUMNS).printSchema()

root
 |-- amount: float (nullable = true)
 |-- month: float (nullable = true)
 |-- day: float (nullable = true)
 |-- day_of_week: float (nullable = true)
 |-- minute: float (nullable = true)



In [7]:
fraud_0 = df.filter(F.col('fraud') == 0).limit(10)
fraud_1 = df.filter(F.col('fraud') == 1).limit(10)

balanced_df = fraud_0.union(fraud_1)

balanced_df.limit(5)

transaction_id,datetime,customer_id,terminal_id,amount,time_seconds,time_days,fraud,fraud_scenario,month,day,day_of_week,hour,minute
0,2019-08-22 06:51:03,0,711,70.91,24663.0,0.0,0.0,0,8.0,22.0,5.0,6.0,51.0
1,2019-08-22 05:10:37,0,0,90.55,18637.0,0.0,0.0,0,8.0,22.0,5.0,5.0,10.0
2,2019-08-22 19:05:33,0,753,35.38,68733.0,0.0,0.0,0,8.0,22.0,5.0,19.0,5.0
3,2019-08-22 07:21:33,0,0,80.41,26493.0,0.0,0.0,0,8.0,22.0,5.0,7.0,21.0
4,2019-08-22 09:06:17,1,981,102.83,32777.0,0.0,0.0,0,8.0,22.0,5.0,9.0,6.0


In [8]:
assembler = VectorAssembler(
    inputCols=TRAIN_COLUMNS,
    outputCol='Features'
)

scaler = MinMaxScaler(
    inputCol='Features',      # Входной столбец с признаками
    outputCol='ScaledFeatures',  # Столбец с масштабированными признаками
)

# scaler
dataproc = Pipeline(stages=[
    assembler
])

In [9]:
ready_data = dataproc.fit(df).transform(df)
ready_data.limit(5).select('Features').limit(1).collect()

[Row(Features=DenseVector([70.91, 8.0, 22.0, 5.0, 51.0]))]

## Подбираем гиперпараметры и тренируем модель

In [15]:
# Определяем пространство поиска для hyperopt
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from pyspark.ml.classification import LogisticRegression

search_space = {
    'regParam': hp.lognormal('regParam', 0, 1.0),
}

def objective(params, train_data, test_data):
    print(params)

    lr = LogisticRegression()\
        .setMaxIter(10)\
        .setRegParam(params['regParam'])\
        .setFeaturesCol('Features')\
        .setLabelCol('fraud')\
        .setFamily("binomial")\

    evaluator = BinaryClassificationEvaluator()\
            .setLabelCol('fraud')

    lg_model = lr.fit(train_data)
    print(f"Coefficients: {lg_model.coefficients}\nIntercept: {lg_model.intercept}")
    print(f"accuracy: {lg_model.summary.accuracy}")
    print(f"weighted Precision: {lg_model.summary.weightedPrecision}")

    auc = evaluator.evaluate(lg_model.transform(test_data))

    with mlflow.start_run():
        mlflow.log_params(params)
        mlflow.log_metric('auc', auc)
    
    return {'loss': -auc, 'status': STATUS_OK}

In [3]:
# from functools import partial

# train_data, test_data = ready_data.randomSplit([.7, .3])
# trials = Trials()

# # # mlflow.set_experiment('classification')

# best = fmin(
#     fn=partial(
#         objective, 
#         train_data=train_data,
#         test_data=test_data
#     ),
#     space=search_space,
#     algo=tpe.suggest,
#     max_evals=1,
#     trials=trials
# )

In [ ]:
def getBestModelfromTrials(trials):
    valid_trial_list = [trial for trial in trials
                            if STATUS_OK == trial['result']['status']]
    losses = [ float(trial['result']['loss']) for trial in valid_trial_list]
    index_having_minumum_loss = np.argmin(losses)
    best_trial_obj = valid_trial_list[index_having_minumum_loss]
    return best_trial_obj['result']['Trained_Model']


best_model = getBestModelfromTrials(trials)


In [ ]:
best_model

In [ ]:
EXPERIMENT_NAME = 'classification'

URI = os.enviroment['MLFLOW_TRACKING_URI']
model_name_mlflow = "best-fraud-detection" 

mlflow.set_tracking_uri(URI)
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(description=RUN_DESCRIPTION) as run:
    mlflow.spark.log_model(best_model, artifact_path="models", registered_model_name=model_name_mlflow)
    print(f"Saved/registered in Run ID: {run.info.run_id}")
    mlflow.end_run()


In [ ]:
import logging
import os
import warnings
import mlflow
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import SparkSession
from scipy.stats import ttest_ind

# Импортирование своих функций

# Отключение предупреждений
warnings.filterwarnings(action='ignore', category=DeprecationWarning)

# Загрузка переменных из .env файла
from dotenv import load_dotenv
load_dotenv()

EXPERIMENT_NAME = 'fraud-classification'

# Задайте переменные для теста
data_path_ab_test = '/Users/admin/PycharmProjects/fraud_detection1/fraud_detection/data'  # Путь к данным
mlflow_uri = 'http://your.mlflow.server:5000'  # URI вашего сервера MLflow
model_name = 'fraud-detection_v1'  # Имя модели

# Функции для Bootstrap выборки
def generate_bootstrap_samples(data, n_iterations=10, sample_fraction=0.5):
    samples = []
    for _ in range(n_iterations):
        samples.append(data.sample(withReplacement=True, fraction=sample_fraction))
    return samples

# Функция для вычисления метрик
def calculate_metrics(model, samples):
    evaluator = BinaryClassificationEvaluator(labelCol="fraud", metricName="areaUnderROC")
    metrics = []

    for sample in samples:
        predictions = model.transform(sample)
        auc = evaluator.evaluate(predictions)
        metrics.append(auc)

    return metrics

# Загрузка модели из MLflow
def load_model_from_mlflow(model_name):
    model_uri = f"models:/{model_name}/1"  # Заменить версию на актуальную
    model = mlflow.spark.load_model(model_uri)
    return model

# Логирование результатов в MLflow
def log_results_to_mlflow(ab_results, bootstrap_results, test_params):
    with mlflow.start_run():
        mlflow.log_params(test_params)
        mlflow.log_metrics({
            "t_stat": ab_results["t_stat"],
            "p_value": ab_results["p_value"]
        })
        mlflow.log_dict(bootstrap_results, "bootstrap_results.json")

# Настройка логирования
logging.basicConfig(level=logging.INFO, format="%(asctime)-15s %(message)s")
logger = logging.getLogger()

logger.info("Creating Spark Session ...")
spark = SparkSession \
    .builder \
    .appName('Spark ML A/B test') \
    .config('spark.sql.repl.eagerEval.enabled', True) \
    .getOrCreate()

mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.set_tracking_uri(mlflow_uri)

logger.info(f"Loading model from MLflow: {model_name} ...")
model = load_model_from_mlflow(model_name)

logger.info(f"Reading data from {data_path_ab_test} ...")
df = spark.read.text(f"{data_path_ab_test}/*.txt")

df = df.sample(fraction=0.0001)

df = clean_data(df)

logger.info("Starting data preparation pipeline ...")
dataproc = data_prep_pipeline()
ready_data = dataproc.fit(df).transform(df)

logger.info("Generating bootstrap samples for group A ...")
bootstrap_samples_a = generate_bootstrap_samples(ready_data)

bootstrap_samples_b = generate_bootstrap_samples(ready_data)

logger.info("Calculating metrics for group A ...")
metrics_a = calculate_metrics(model, bootstrap_samples_a)

logger.info("Calculating metrics for group B ...")
metrics_b = calculate_metrics(model, bootstrap_samples_b)

logger.info("Performing A/B testing ...")
t_stat, p_value = ttest_ind(metrics_a, metrics_b)
logger.info(f"A/B Test completed: t_stat={t_stat}, p_value={p_value}")

logger.info("Logging results to MLflow ...")
log_results_to_mlflow(
    ab_results={"t_stat": t_stat, "p_value": p_value},
    bootstrap_results={"metrics_a": metrics_a, "metrics_b": metrics_b},
    test_params={"model_name": "fraud_model"}
)
logger.info("Results successfully logged to MLflow.")

spark.stop()
